[Guide Index](README.md)


# 05 StarLayer Schemas and Ontologies

StarLayer provides the capability to parse text inputs into Python objects for use with RDF graphs.  This includes SPARQL queries, SPARQL rules (SRL), and Manchester (OWL) text.  

StarLayer also provides the capability to parse these same text files into RDF graphs using specific StarLayer ontologies.  Managing text files as RDF graphs allows for their manipulation as graphs, SHACL validation of the graphs and for their use with SHACL-based viewers and editors (SHACL-UI).   These are referred to as "Graph Trees" because they are designed to capture the tree-like structure of the text file. They are supported with ontology and SHACL files.

- **[Manchester Syntax](05a-manchester-syntax.ipynb)** (`manch:`) - (OWL 2)  
- **[SPARQL queries as RDF](05b-sparql-query-as-rdf.ipynb)** (`sqe:`) 
- **[SRL (SPARQL Rules) as RDF](05c-srl-sparql-rules-as-rdf.ipynb)** (`srl:`) 

StarLayer provides similar support for SKOS files (`skos:`).  However, as they are already RDF graphs in a tree-like form, no special parsing function is needed.  SKOS graphs are supported with ontology and SHACL files.

The rest of this guide covers the registry that bundles all of these vocabularies - ontology and SHACL files, how to load them, validate against them, and fetch them directly.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.


In [1]:
from starlayer import starontology

## 1. List what's registered

The syntax-tree graphs are supported with ontology files.  The list of available ontology files can be obtained using the `get_ontology_list()` function.  

You will see below ontology (owl) and shacl files for manchester, sparql (sqe), sparql rules (srl) and skos.  spexpr files are imported into sqe and srl but not used on their own.  shacl-meta is a shacl shape graph used to validate other shacl shape graphs.

In [2]:
for info in starontology.get_ontology_list():
    print(f"{info.name:20} {info.type:10} {info.description[:80]}")

manchester_owl       syntax-tree-graph OWL 2 Manchester Syntax's own internal syntax tree, as RDF (manch:) - frames/cla
manchester_shacl     shacl      SHACL shapes validating a manch: syntax tree's own structural well-formedness.
shacl_meta           shacl      The SHACL 1.2 meta-shapes themselves (starlayer.shacl's own shapes-about-shapes,
skos_owl             owl        The W3C SKOS Reference's own formal axioms and numbered Documented Consistency a
skos_shacl           shacl      SHACL shapes checking SKOS's own numbered integrity conditions against a data gr
spexpr_owl           syntax-tree-graph The SPARQL expression/variable/property-path vocabulary (spexpr:), carved out of
spexpr_shacl         shacl      SHACL shapes validating an spexpr: expression/variable/path value's own well-for
sqe_owl              syntax-tree-graph A SPARQL SELECT/ASK/CONSTRUCT query's own raw parse tree, pre-translateQuery, as
sqe_shacl            shacl      SHACL shapes validating an sqe: query-edit gr

## 2. Load an ontology graph

`get_ontology_graph(name)` returns a fresh `StarLayerGraph` of the named file.  

In [3]:
skos_owl = starontology.get_ontology_graph("skos_owl")
skos_shacl = starontology.get_ontology_graph("skos_shacl")
print("skos_owl:  ", len(skos_owl), "triples")
print("skos_shacl:", len(skos_shacl), "triples")

skos_owl:   90 triples
skos_shacl: 102 triples


## 3. Validate against a registered shapes graph

Any of the registered `*_shacl` entries works directly with `StarShaclSchema` to validate a data graph.

In [4]:
from starlayer import StarLayerGraph, StarShaclSchema

# load a skos file
skos = StarLayerGraph()
skos.parse(data="""
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
    @prefix ex: <http://example.org/> .
    ex:scheme1 a skos:ConceptScheme .
    ex:concept1 a skos:Concept ; skos:inScheme ex:scheme1 ; skos:prefLabel "Example"@en .
""", format="turtle")

# obtain the skos shacl file.
skos_shacl = starontology.get_ontology_graph("skos_shacl")

# create a ShaclScema object and validate
skosShaclSchema = StarShaclSchema()
skosShaclSchema.shacl_graph = skos_shacl
result = skosShaclSchema.validate(data_graph=skos, meta_shacl=False)
print("conforms:", result.conforms)

conforms: True


## 4. Conformance with a specification or a standard

As described in [SHACL Profiling guide](04e-shacl-profiling.ipynb), a data graph that is validated by a shapes graph `sh:conformsTo` that shapes graph.  If that shapes graph in turn asserts that it represents a profile of a given given standard, then the data graph `dc:conformsTo` the given standard.   

Starlayer provides a profile document `ontology-shapes-profiles.ttl` that contains profiles of various standards and the mapping to the relevant shacl shape graph.  This allows for the assertion that a graph complies with the relevant standard.  
 

In [5]:
# the whole mapping document at once, instead of one pair at a time
profiles_graph = starontology.get_conformance_profiles()
print(profiles_graph.serialize())

@prefix prof: <http://www.w3.org/ns/dx/prof/> .
@prefix profile: <http://www.w3.org/ns/shacl/profile/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix role: <http://www.w3.org/ns/dx/prof/role/> .
@prefix starontology: <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/> .
@prefix tr: <https://www.w3.org/TR/> .

starontology:manchester_shapes.ttl prof:isProfileOf tr:owl2-manchester-syntax .

starontology:shacl_meta_shapes.ttl prof:isProfileOf profile:sparql, profile:node-expr, profile:profiling, profile:core, profile:rules .

starontology:skos_shapes.ttl prof:isProfileOf tr:skos-reference .

starontology:spexpr_shapes.ttl prof:isProfileOf tr:sparql12-query .

starontology:sqe_shapes.ttl prof:isProfileOf tr:sparql12-query .

starontology:srl_shapes.ttl prof:isProfileOf tr:sparql12-rl .

tr:owl2-manchester-syntax rdfs:label "OWL 2 Manchester Syntax" ;
    prof:hasResource [
        a prof:ResourceDescriptor ;
        rdfs:comme

## 5. The full conformance chain

Validating against a registered shapes graph auto-fills the report's `sh:usedShapesGraph` from the shapes graph's own `.identifier` - the same identity `get_conformance_profiles()` asserts `prof:isProfileOf` on. That means `shacl_profiling.derive_conforms_to()`'s second rule can match it directly, deriving a `dcterms:conformsTo` triple all the way through to the real specification, not just `sh:conformsTo` to the shapes graph.

In [6]:
from starlayer.shacl import shacl_profiling

skos_shacl = starontology.get_ontology_graph("skos_shacl")

# the skos graph is validated against the skos shacl file.
# the iri of the data graph is provided.
# the iri of the skos_shacl file is available in the graph object.
result = StarShaclSchema(shacl_graph=skos_shacl).validate(
    data_graph=skos,
    meta_shacl=False,
    data_graph_iri="http://example.org/graphs/my-skos-data",
 
)

# report graph asserts the data graph conforms to the shacl graph
print ("Result of validation:")
print (result.report_graph.serialize())

# derive_confoms checks whether a report graph supports asserting
# that the data graph conforms to the related specificaiton.
derived = shacl_profiling.derive_conforms_to(
    result.report_graph,
    specification_graph=starontology.get_conformance_profiles(),
    conforms_only=True,
)
print ("Result of conformance check:")
print (derived.serialize())


Result of validation:
@prefix ex: <http://example.org/> .
@prefix ns1: <http://example.org/graphs/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix starontology: <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:my-skos-data sh:conformsTo starontology:skos_shapes.ttl ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Violation, sh:Warning, sh:Info ;
        sh:conforms true ;
        sh:usedDataGraph ns1:my-skos-data ;
        sh:usedShapesGraph starontology:skos_shapes.ttl
    ] .

Result of conformance check:
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ex: <http://example.org/> .
@prefix ns1: <http://example.org/graphs/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix starontology: <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/> .
@prefix tr: <https://www.w3.org/TR/>

## Further reading

Every registry entry above is also fetchable directly from GitHub, outside Python entirely (by stable identity, or by raw content) - see [Appendix A: Public Resources](appendix-a-public-resources.ipynb).